# GECA Twitch · Exploración transaccional y agregados
Abre desde la raíz del proyecto, selecciona el kernel Conda `geca-twitch`. No descarga videos ni usa LLM.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

DB = Path("state/geca_twitch.sqlite3")
if not DB.exists():
    DB = Path("../state/geca_twitch.sqlite3")
assert DB.exists(), f"No se encontró {DB}: primero ejecuta ingest.py"
conn = sqlite3.connect(DB)
print(DB.resolve())

## 1. Tablas y volúmenes

In [ ]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name", conn)["name"].tolist()
counts = []
for name in tables:
    counts.append({"tabla":name,"registros":conn.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]})
display(pd.DataFrame(counts))

## 2. Catálogo Steam + Twitch e IGDB

In [ ]:
display(pd.read_sql_query("SELECT steam_appid,steam_name,twitch_game_id,twitch_name,igdb_id,status,confidence FROM (SELECT s.steam_appid,s.steam_name,m.twitch_game_id,t.twitch_name,t.igdb_id,m.status,m.confidence FROM steam_games s LEFT JOIN game_matches m ON s.steam_appid=m.steam_appid LEFT JOIN twitch_games t ON t.twitch_game_id=m.twitch_game_id) LIMIT 30",conn))

## 3. Transmisiones individuales

In [ ]:
display(pd.read_sql_query("SELECT captured_utc,twitch_game_id,stream_id,user_name,user_login,viewer_count,language,title,tags_json FROM stream_snapshots ORDER BY captured_utc DESC LIMIT 50",conn))

## 4. Fotografías de audiencia

In [ ]:
display(pd.read_sql_query("SELECT g.captured_utc,t.twitch_name,g.viewers_observed,g.streams_observed,g.streams_complete FROM game_snapshots g LEFT JOIN twitch_games t USING(twitch_game_id) ORDER BY g.captured_utc DESC LIMIT 50",conn))

## 5. Videos: enlaces y últimas vistas observadas, sin descarga

In [ ]:
display(pd.read_sql_query("SELECT v.video_id,g.twitch_name,v.title,v.url,v.duration,v.user_name,o.view_count,o.captured_utc FROM twitch_videos v LEFT JOIN twitch_games g USING(twitch_game_id) LEFT JOIN video_observations o ON o.video_id=v.video_id AND o.rowid=(SELECT x.rowid FROM video_observations x WHERE x.video_id=v.video_id ORDER BY x.captured_utc DESC LIMIT 1) LIMIT 50",conn))

## 6. Clips: enlaces, creadores y última observación

In [ ]:
display(pd.read_sql_query("SELECT c.clip_id,g.twitch_name,c.title,c.url,c.creator_name,c.is_featured,o.view_count FROM twitch_clips c LEFT JOIN twitch_games g USING(twitch_game_id) LEFT JOIN clip_observations o ON o.clip_id=c.clip_id AND o.rowid=(SELECT x.rowid FROM clip_observations x WHERE x.clip_id=c.clip_id ORDER BY x.captured_utc DESC LIMIT 1) LIMIT 50",conn))

## 7. Agregados diarios y validación de cobertura

In [ ]:
display(pd.read_sql_query("SELECT * FROM daily_metrics ORDER BY utc_day DESC,avg_viewers DESC LIMIT 50",conn))

## 8. Por qué no se activa una alerta

In [ ]:
display(pd.read_sql_query("SELECT e.*,g.twitch_name FROM attention_evaluations e LEFT JOIN twitch_games g USING(twitch_game_id) ORDER BY evaluated_utc_day DESC LIMIT 50",conn))

## 9. Propuestas editoriales y decisiones humanas

In [ ]:
display(pd.read_sql_query("SELECT * FROM editorial_proposals ORDER BY proposal_day DESC LIMIT 30",conn))
display(pd.read_sql_query("SELECT * FROM editorial_decisions ORDER BY decided_utc DESC LIMIT 30",conn))

## 10. Recepción Steam (NO sentimiento de Twitch)

In [ ]:
display(pd.read_sql_query("SELECT steam_name,reviews_total,reviews_positive,reviews_negative FROM steam_games ORDER BY reviews_total DESC LIMIT 30",conn))
conn.close()